# Entrenamiento de YOLO11 Nano Segment en Google Colab con Google Drive

Este cuaderno contiene la estructura para entrenar el modelo de segmentación **YOLO11 Nano (`yolo11n-seg`)** conectando tu **Google Drive** para cargar el dataset de forma persistente y guardar los pesos finales de forma automática para no perderlos.

### Instrucciones previas:
1. Comprime la carpeta de tu dataset convertido (ej. `datasetpesadopesadon` o `Food types.v3i.yolo-segmentation`) en un archivo ZIP llamado `datasetpesadopesadon.zip`.
2. Sube ese `datasetpesadopesadon.zip` a tu cuenta de Google Drive (ej. en la raíz de tu Drive o en una carpeta de tu preferencia).
3. Asegúrate de activar el entorno de ejecución con **GPU** en Colab: `Entorno de ejecución` > `Cambiar tipo de entorno de ejecución` > Seleccionar **T4 GPU** o superior.

## Paso 1: Conectar Google Drive

In [ ]:
from google.colab import drive
import os

# Montar Google Drive en '/content/drive'
drive.mount('/content/drive')

print("Google Drive conectado correctamente.")

## Paso 2: Instalar dependencias y verificar GPU

In [ ]:
# Instalar la librería Ultralytics para YOLO11
%pip install ultralytics

import torch
from ultralytics import YOLO, checks

# Verificar recursos asignados
checks()
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo activo: {torch.cuda.get_device_name(0)}")

## Paso 3: Copiar y Descomprimir el Dataset en el Disco Local de Colab

> **Nota importante de rendimiento:** Copiar y descomprimir el archivo ZIP en el almacenamiento local de Colab (`/content/datasetpesadopesadon`) en lugar de entrenar directamente sobre los archivos montados de Google Drive acelera dramáticamente la velocidad del entrenamiento y evita problemas de límite de solicitudes en la API de Drive.

In [ ]:
# Define la ruta exacta de tu archivo zip dentro de Google Drive
# Por defecto, si está en la raíz de tu Drive, la ruta será: '/content/drive/MyDrive/datasetpesadopesadon.zip'
zip_path_in_drive = '/content/drive/MyDrive/datasetpesadopesadon.zip'

local_dataset_dir = '/content/datasetpesadopesadon'

if os.path.exists(zip_path_in_drive):
    print(f"Copiando y descomprimiendo {zip_path_in_drive} a almacenamiento local...")
    # Descomprimir directamente al almacenamiento rápido local de Colab
    !unzip -q {zip_path_in_drive} -d {local_dataset_dir}
    print("Descompresión completada con éxito.")
else:
    print(f"Error: No se encontró el archivo '{zip_path_in_drive}' en Google Drive.")
    print("Por favor verifica el nombre del archivo y que esté ubicado en tu carpeta de Drive.")

## Paso 4: Ajustar rutas en `data.yaml` para Colab

In [ ]:
import yaml

yaml_path = '/content/datasetpesadopesadon/data.yaml'

if os.path.exists(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    
    # Ajustar campo path al directorio local donde se descomprimió el dataset
    data['path'] = '/content/datasetpesadopesadon'
    
    with open(yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, sort_keys=False)
        
    print("data.yaml actualizado con éxito para Colab:")
    print(yaml.dump(data))
else:
    print("Error: No se encontró el archivo data.yaml. Verifica la ruta de descompresión.")

## Paso 5: Entrenar el Modelo YOLO11 Segment

In [ ]:
# Inicializar pesos del modelo de segmentación YOLO11 Nano
model = YOLO('yolo11n-seg.pt')

# Entrenar el modelo
results = model.train(
    data='/content/datasetpesadopesadon/data.yaml',
    epochs=100,                  # Número de épocas
    imgsz=640,                   # Tamaño de imagen
    batch=16,                    # Tamaño de lote
    device=0,                    # GPU
    project='datasetpesadopesadon_project',  # Directorio de salida
    name='yolo11n_seg',      # Nombre del experimento
    save=True,
    plots=True
)

## Paso 6: Visualizar Resultados

In [ ]:
from IPython.display import Image, display

results_img_path = '/content/datasetpesadopesadon_project/yolo11n_seg/results.png'
if os.path.exists(results_img_path):
    print("Métricas de Entrenamiento:")
    display(Image(filename=results_img_path))
else:
    print("El archivo results.png no se generó o el entrenamiento no se completó.")

## Paso 7: Guardar Resultados y Pesos Persistentes en Google Drive

Para evitar perder tus pesos entrenados (`modelofinal.pt`) y gráficas al expirar o cerrarse el entorno temporal de Colab, los respaldaremos automáticamente en una carpeta dentro de tu Google Drive.

In [ ]:
# Carpeta destino de respaldo en Google Drive
drive_backup_dir = '/content/drive/MyDrive/datasetpesadopesadon_project_results'
os.makedirs(drive_backup_dir, exist_ok=True)

print(f"Copiando resultados a {drive_backup_dir}...")

# Copiar los mejores pesos y el archivo de métricas
!cp /content/datasetpesadopesadon_project/yolo11n_seg/weights/best.pt {drive_backup_dir}/modelofinal.pt
!cp /content/datasetpesadopesadon_project/yolo11n_seg/weights/last.pt {drive_backup_dir}/last.pt
!cp /content/datasetpesadopesadon_project/yolo11n_seg/results.png {drive_backup_dir}/results.png

# Comprimir la carpeta completa y guardarla en Drive por si necesitas más información
!zip -r {drive_backup_dir}/datasetpesadopesadon_project_complete.zip /content/datasetpesadopesadon_project

print("\n--- RESPALDO COMPLETADO EN GOOGLE DRIVE ---")
print(f"Los mejores pesos están respaldados en: {drive_backup_dir}/modelofinal.pt")